In [3]:
import os
from google.api_core.client_options import ClientOptions
from google.cloud import documentai_v1 as documentai
from typing import Dict, Any
from finance_analysis.config import global_config as glob   

def process_invoice_raw(
    project_id: str,
    location: str,
    processor_id: str,
    file_path: str,
    mime_type: str = "application/pdf",
) -> Dict[str, Any]:
    """
    Call Document AI (Invoice Parser or Enterprise OCR) and return the raw Document as a dict.
    """
    client_options = ClientOptions(api_endpoint=f"{location}-documentai.googleapis.com")
    client = documentai.DocumentProcessorServiceClient(client_options=client_options)

    name = client.processor_path(project_id, location, processor_id)

    with open(file_path, "rb") as f:
        file_content = f.read()

    raw_document = documentai.RawDocument(content=file_content, mime_type=mime_type)

    request = documentai.ProcessRequest(
        name=name,
        raw_document=raw_document,
    )

    result = client.process_document(request=request)

    # This is the full raw protobuf object:
    proto_doc = result.document

    # Option 1: keep it as protobuf (you can inspect fields directly)
    # Option 2: convert to a Python dict (JSON-like)
    document_dict = documentai.Document.to_dict(proto_doc)

    return document_dict


In [4]:
invoice_name = "Free2move_DE_251113_4248127_3285000079177057.pdf"
invoice_name = "Departure_taxi.pdf"
invoice_name = "Hotel_Invoice.pdf"

doc = process_invoice_raw(
    project_id="neme-ai-rnd-dev-prj-01",
    location="eu",
    processor_id="c48f6c912b9ff9d5",
    file_path=os.path.join(glob.DATA_PKG_DIR, invoice_name)
)

In [5]:
doc

{'document_layout': {'blocks': [{'block_id': '1',
    'text_block': {'text': 'HOTELS', 'type_': 'paragraph', 'blocks': []},
    'page_span': {'page_start': 1, 'page_end': 1}},
   {'block_id': '2',
    'text_block': {'text': 'DANUBIUS', 'type_': 'paragraph', 'blocks': []},
    'page_span': {'page_start': 1, 'page_end': 1}},
   {'block_id': '3',
    'text_block': {'text': 'Számla/Invoice',
     'type_': 'paragraph',
     'blocks': []},
    'page_span': {'page_start': 1, 'page_end': 1}},
   {'block_id': '4',
    'text_block': {'text': 'Oldal / Page', 'type_': 'paragraph', 'blocks': []},
    'page_span': {'page_start': 1, 'page_end': 1}},
   {'block_id': '5',
    'text_block': {'text': '1/2', 'type_': 'paragraph', 'blocks': []},
    'page_span': {'page_start': 1, 'page_end': 1}},
   {'block_id': '6',
    'table_block': {'body_rows': [{'cells': [{'blocks': [{'block_id': '7',
           'text_block': {'text': 'VOSSELER, Name/Név:',
            'type_': 'paragraph',
            'blocks': []},

# Comparison: Docling vs Document AI

Let's compare two approaches for extracting invoice data:
1. **Docling** (current approach) - OCR + Layout parsing with Docling library
2. **Google Document AI** - Cloud-based document processing with Layout Parser

In [6]:
# Test the same invoice with Docling (current approach)
from finance_analysis.resources.document_processor import DocumentProcessor

# Use the same invoice file
docling_processor = DocumentProcessor(
    file_path=os.path.join(glob.DATA_PKG_DIR, invoice_name)
)

# Process the document
docling_doc = docling_processor.process()

2025-11-16 01:11:18,411 - finance_analysis.resources.document_processor - INFO - ✅ Document converted successfully: Hotel_Invoice.pdf


In [7]:
docling_doc.export_to_markdown()

'Name/Név:\n\n1 / 2\n\nOldal / Page\n\n## Számla/Invoice\n\nName and address of partner/Vevô neve, címe:\n\nVOSSELER, ALEXANDER\n\n09.11.25\n\nArrival/Érkezés:\n\n13.11.25\n\nDeparture/Utazás:\n\n10401227126\n\nInvoice No/Számlaszám:\n\nRoom No./Szobaszám:\n\n230\n\nReference/Ügyintézô:\n\nPeter Vieland\n\nDate of Perf./Teljesités dátuma:\n\nInvoice date/Számla kelte:\n\n13.11.25\n\n13.11.25\n\nDue date/Fizetési határidô:\n\n13.11.25\n\nRef./Hiv.:\n\nNemetschek Group SE Konrad-Zuse-Platz 1 München, Bayern  81829 Germany\n\nTax code/Adószám:\n\n| Date of services/Szolg. dátum   | Service/Szolgáltatás Product/Termék (Stat. No/SZJ Vat/Áfa %)   | Copy M. / Q. M.e. / U. of q.   | Net/Nettó:          | Net/Nettó:        |                       | Gross HUF          |\n|---------------------------------|----------------------------------------------------------------|--------------------------------|---------------------|-------------------|-----------------------|--------------------|\n| Date

In [ ]:
# Display Docling extracted content as markdown
docling_processor.display_markdown()

Name/Név:

1 / 2

Oldal / Page

## Számla/Invoice

Name and address of partner/Vevô neve, címe:

VOSSELER, ALEXANDER

09.11.25

Arrival/Érkezés:

13.11.25

Departure/Utazás:

10401227126

Invoice No/Számlaszám:

Room No./Szobaszám:

230

Reference/Ügyintézô:

Peter Vieland

Date of Perf./Teljesités dátuma:

Invoice date/Számla kelte:

13.11.25

13.11.25

Due date/Fizetési határidô:

13.11.25

Ref./Hiv.:

Nemetschek Group SE Konrad-Zuse-Platz 1 München, Bayern  81829 Germany

Tax code/Adószám:

| Date of services/Szolg. dátum   | Service/Szolgáltatás Product/Termék (Stat. No/SZJ Vat/Áfa %)   | Copy M. / Q. M.e. / U. of q.   | Net/Nettó:          | Net/Nettó:        |                       | Gross HUF          |
|---------------------------------|----------------------------------------------------------------|--------------------------------|---------------------|-------------------|-----------------------|--------------------|
| Date of services/Szolg. dátum   | Service/Szolgáltatás Product/Termék (Stat. No/SZJ Vat/Áfa %)   | Copy M. / Q. M.e. / U. of q.   | Unit Price Egységár | Net Flat Összesen | VAT Amount ÁFA összeg | Value Bruttó Érték |
| 09.11.25                        | Visa Debit                                                     | 1 db                           |                     |                   |                       | -252,050           |
| 09.11.25                        | -Room/Szoba 55.10.1 5%                                         | 1 db                           | 52,844              | 52,844            | 2,642                 | 55,486             |
| 09.11.25                        | -City Tax / IFA                                                | 1 db                           | 2,114               | 2,114             |                       | 2,114              |
| 09.11.25                        | -Package Breakfast/Reggeli 56.10.11 5%                         | 1 db                           | 5,409               | 5,409             | 270                   | 5,679              |
| 09.11.25                        | -Service Charge/Felszolgálási díj 56.10.11 5%                  | 1 db                           | 114                 | 114               | 6                     | 120                |
| 10.11.25                        | -Room/Szoba 55.10.1 5%                                         | 1 db                           | 56,746              | 56,746            | 2,837                 | 59,583             |
| 10.11.25                        | -City Tax / IFA                                                | 1 db                           | 2,270               | 2,270             |                       | 2,270              |
| 10.11.25                        | -Package Breakfast/Reggeli 56.10.11 5%                         | 1 db                           | 5,409               | 5,409             | 270                   | 5,679              |
| 10.11.25                        | -Service Charge/Felszolgálási díj 56.10.11 5%                  | 1 db                           | 114                 | 114               | 6                     | 120                |
| 11.11.25                        | -Room/Szoba 55.10.1 5%                                         | 1 db                           | 51,071              | 51,071            | 2,554                 | 53,625             |
| 11.11.25                        | -City Tax / IFA                                                | 1 db                           | 2,042               | 2,042             |                       | 2,042              |
| 11.11.25                        | -Package Breakfast/Reggeli 56.10.11 5%                         | 1 db                           | 5,409               | 5,409             | 270                   | 5,679              |
| 11.11.25                        | -Service Charge/Felszolgálási díj 56.10.11 5%                  | 1 db                           | 114                 | 114               | 6                     | 120                |
| 12.11.25                        | -Room/Szoba 55.10.1 5%                                         | 1 db                           | 49,297              | 49,297            | 2,465                 | 51,762             |
| 12.11.25                        | -City Tax / IFA                                                | 1 db                           | 1,972               | 1,972             |                       | 1,972              |
| 12.11.25                        | -Package Breakfast/Reggeli 56.10.11 5%                         | 1 db                           | 5,409               | 5,409             | 270                   | 5,679              |
| 12.11.25                        | -Service Charge/Felszolgálási díj 56.10.11 5%                  | 1 db                           | 114                 | 114               | 6                     | 120                |

<!-- image -->

Name/Név:

2 / 2

Oldal / Page

## Számla/Invoice

Name and address of partner/Vevô neve, címe:

VOSSELER, ALEXANDER

09.11.25

Arrival/Érkezés:

13.11.25

Departure/Utazás:

10401227126

Invoice No/Számlaszám:

Room No./Szobaszám:

230

Reference/Ügyintézô:

Peter Vieland

Date of Perf./Teljesités dátuma:

Invoice date/Számla kelte:

13.11.25

13.11.25

Due date/Fizetési határidô:

13.11.25

Ref./Hiv.:

Nemetschek Group SE Konrad-Zuse-Platz 1 München, Bayern  81829 Germany

Tax code/Adószám:

| Date of services/Szolg.   | Service/Szolgáltatás                 |             | M. / Net/Nettó:   | Q.                  | Net Flat Összesen          | Gross HUF                  | Value        |
|---------------------------|--------------------------------------|-------------|-------------------|---------------------|----------------------------|----------------------------|--------------|
| dátum                     | Product/Termék (Stat. No/SZJ Vat/Áfa | %)          | M.e. / U. Price   | of q. Unit Egységár |                            | VAT Amount ÁFA összeg      | Bruttó Érték |
|                           |                                      | Nettó / Net | ÁFA / VAT         | Bruttó / Gross      | Amount Due/Fizetendô (HUF) | Amount Due/Fizetendô (HUF) | 0            |
| VAT 27%/ÁFA 27%           | VAT 27%/ÁFA 27%                      | 0           | 0                 | 0                   |                            |                            |              |
| VAT 18%/ÁFA 18%           | VAT 18%/ÁFA 18%                      | 0           | 0                 | 0                   |                            | TIP (HUF)                  |              |
| VAT 5%/ÁFA 5%             | VAT 5%/ÁFA 5%                        | 232,050     | 11,602            | 243,652             |                            |                            |              |
| VAT 0%/ÁFAmentes          | VAT 0%/ÁFAmentes                     | 0           | 0                 | 0                   |                            |                            |              |
| No VAT/ÁFA kör.kív.       | No VAT/ÁFA kör.kív.                  | 0           | 0                 | 0                   |                            |                            |              |
| Total/Összesen            | Total/Összesen                       | 232,050     | 11,602            | 243,652             |                            |                            |              |
| City Taxl/IFA             | City Taxl/IFA                        | 8,398       | 0                 | 8,398               |                            |                            |              |
| Total/Mindösszesen        | Total/Mindösszesen                   | 240,448     | 11,602            | 252,050             |                            |                            |              |

Comments/Megjegyzések:

Copy M. / Q. Unit Price Egységár 8,398

Guest Signature/Vendég Aláírás

This invoice was created by Opera V5, Micros Fidelio Inc./A számla az Opera V5, Micros Fidelio Inc. rendszerével készült. Only amounts printed by the cashier are valid!/Csak a számlázógéppel nyomtatott összegek érvényesek! Valid without signature and stamp./Aláírás és pecsét nélkül érvényes. ÁKK - out of scope of VAT/ÁFA tv. hatályán kívül. ÁFA m. - exempt from VAT payment in line with Act on VAT/ÁFA tv. szerint áfa mentes. FAD - reverse charge procedure/Fordított adózás.

| Merchant ID                                                                                            |                                                                                                        | Credit Card #/Kártyaszám:                                                                              | XXXXXXXXXXXX9534                                                                                       |
|--------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|
| Transaction ID                                                                                         | 10453625                                                                                               | Credit Card Expiry/Lejárat                                                                             | XX/XX                                                                                                  |
| Approval Code/Eng.kód                                                                                  | 220314                                                                                                 | Capture Method :                                                                                       | Chip                                                                                                   |
| Approval Amount/Eng.összeg                                                                             | 252,050                                                                                                | Amount/Összeg                                                                                          | 252,050                                                                                                |
| Please charge the above amount to my credit card/Kérem a fenti összeggel a hitelkártyámat megterhelni. | Please charge the above amount to my credit card/Kérem a fenti összeggel a hitelkártyámat megterhelni. | Please charge the above amount to my credit card/Kérem a fenti összeggel a hitelkártyámat megterhelni. | Please charge the above amount to my credit card/Kérem a fenti összeggel a hitelkártyámat megterhelni. |

<!-- image -->

## Document AI Output as Markdown

**Note:** Document AI Layout Parser returns raw block layout without semantic table structure (no header/data row distinction), resulting in messy tables. This demonstrates why Docling is better for invoice processing.

In [ ]:
from IPython.display import Markdown, display

def docai_to_markdown(doc_dict):
    """
    Convert Document AI blocks to clean markdown format.
    
    Processes text blocks and table blocks separately, formatting tables
    with proper headers and maintaining document structure.
    """
    lines, table_rows = [], []
    
    # Iterate through all layout blocks in the document
    for block in doc_dict.get('document_layout', {}).get('blocks', []):
        
        # Handle text blocks (paragraphs, headings)
        if 'text_block' in block and (text := block['text_block'].get('text', '').strip()):
            # Flush any accumulated table rows before adding text
            if table_rows:
                lines.extend(_format_table(table_rows))
                table_rows = []
            
            # Determine block type and add appropriate markdown prefix
            block_type = block['text_block'].get('type_', 'paragraph')
            prefix = '\n# ' if block_type == 'heading-1' else '\n## ' if block_type == 'heading-2' else ''
            lines.append(f"{prefix}{text}\n")
        
        # Handle table blocks - collect rows for batch processing
        elif 'table_block' in block:
            for row in block['table_block'].get('body_rows', []):
                # Extract text from each cell, joining multiple text blocks within a cell
                row_data = [' '.join(cb['text_block'].get('text', '').strip() 
                           for cb in cell.get('blocks', []) if 'text_block' in cb) or '-'
                           for cell in row.get('cells', [])]
                # Only keep rows that have at least one non-empty cell
                if any(r != '-' for r in row_data):
                    table_rows.append(row_data)
    
    # Flush any remaining table rows at the end
    if table_rows:
        lines.extend(_format_table(table_rows))
    
    return "\n".join(lines)

def _format_table(rows):
    """
    Format rows as a proper markdown table with auto-detected headers.
    
    Args:
        rows: List of lists containing cell data
        
    Returns:
        List of formatted markdown table lines
    """
    # Normalize all rows to have the same number of columns
    max_cols = max(len(r) for r in rows)
    rows = [r + ['-'] * (max_cols - len(r)) for r in rows]
    
    # Auto-detect header: use first row if it's mostly populated
    has_header = len(rows) > 1 and sum(bool(c != '-') for c in rows[0]) >= max_cols // 2
    header = rows[0] if has_header else [f"Col{i+1}" for i in range(max_cols)]
    data = rows[1:] if has_header else rows
    
    # Build markdown table with header, separator, and data rows
    return [
        "\n| " + " | ".join(header) + " |",           # Header row
        "| " + " | ".join(['---'] * max_cols) + " |",  # Separator
        *["| " + " | ".join(r) + " |" for r in data],  # Data rows
        ""  # Blank line after table
    ]

# Convert and display the Document AI output as formatted markdown
display(Markdown(docai_to_markdown(doc)))

HOTELS

DANUBIUS

Számla/Invoice

Oldal / Page

1/2


| VOSSELER, Name/Név: | ALEXANDER |
| --- | --- |
| Ref./Hiv.: | - |
| Arrival/Érkezés: | 09.11.25 |
| Departure/Utazás: | 13.11.25 |
| Room No./Szobaszám: | 230 |
| Invoice date/Számla kelte: | 13.11.25 |
| Date of Perf./Teljesités dátuma: | 13.11.25 |
| Due date/Fizetési határidô: | 13.11.25 |
| Invoice No/Számlaszám: | 10401227126 |
| Reference/Ügyintézô: | Peter Vieland |

Name and address of partner/Vevô neve, címe:

Nemetschek Group SE

Konrad-Zuse-Platz 1

München, Bayern 81829 Germany

Tax code/Adószám:


| Date of | Service/Szolgáltatás | Net/Nettó: | - | - | - | HUF |
| --- | --- | --- | --- | --- | --- | --- |
| services/Szolg. dátum | M. / Q. M.e. / U. of q. Product/Termék (Stat. No/SZJ Vat/Áfa %) | Unit Price Egységár | - | Net Flat Összesen | VAT Amount ÁFA összeg | Gross Value Bruttó Érték |
| 09.11.25 | Visa Debit 1 db | - | - | - | - | -252,050 |
| 09.11.25 | -Room/Szoba 55.10.15% 1 db | 52,844 | - | 52,844 | 2,642 | 55,486 |
| 09.11.25 | -City Tax / IFA 1 db | 2,114 | - | 2,114 | - | 2,114 |
| 09.11.25 | -Package Breakfast/Reggeli 56.10.11 5% 1 db | 5,409 | - | 5,409 | 270 | 5,679 |
| 09.11.25 | -Service Charge/Felszolgálási díj 56.10.11 5% 1 db | 114 | - | 114 | 6 | 120 |
| 10.11.25 | -Room/Szoba 55.10.15% 1 db | 56,746 | - | 56,746 | 2,837 | 59,583 |
| 10.11.25 | -City Tax / IFA 1 db | 2,270 | - | 2,270 | - | 2,270 |
| 10.11.25 | -Package Breakfast/Reggeli 56.10.11 5% 1 db | 5,409 | - | 5,409 | 270 | 5,679 |
| 10.11.25 | -Service Charge/Felszolgálási díj 56.10.11 5% 1 db | 114 | - | 114 | 6 | 120 |
| 11.11.25 | -Room/Szoba 55.10.15% 1 db | 51,071 | - | 51,071 | 2,554 | 53,625 |
| 11.11.25 | -City Tax / IFA 1 db | 2,042 | - | 2,042 | - | 2,042 |
| 11.11.25 | -Package Breakfast/Reggeli 56.10.11 5% 1 db | 5,409 | - | 5,409 | 270 | 5,679 |
| 11.11.25 | -Service Charge/Felszolgálási díj 56.10.11 5% 1 db | 114 | - | 114 | 6 | 120 |
| 12.11.25 | -Room/Szoba 55.10.15% 1 db | 49,297 | - | 49,297 | 2,465 | 51,762 |
| 12.11.25 | -City Tax / IFA 1 db | 1,972 | - | 1,972 | - | 1,972 |
| 12.11.25 | -Package Breakfast/Reggeli 56.10.11 5% 1 db | 5,409 | - | 5,409 | 270 | 5,679 |
| 12.11.25 | -Service Charge/Felszolgálási díj 56.10.11 5% 1 db | 114 | - | 114 | 6 | 120 |

Danubius Hotels Zrt.

Danubius Hotel Helia

H-1133 Budapest, Kárpát u. 62-64.

Tax number/Adószám: 10594702-2-44

EU tax number/Közösségi adószám: HU10594702

Banknév és számlaszámok/Bank's name and accounts:

K&H Bank Zrt., H-1095 Budapest, Lechner Ödön fasor 9.

K&H EUR IBAN: HU29 1040 21424948 49565757 1817

K&H HUF: 10402142-49484956-57571721

SWIFT: OKHBHUHB

HOTELS

H DANUBIUS

Számla/Invoice

Oldal / Page

2/2


| VOSSELER, Name/Név: | ALEXANDER |
| --- | --- |
| Ref./Hiv.: | - |
| Arrival/Érkezés: | 09.11.25 |
| Departure/Utazás: | 13.11.25 |
| Room No./Szobaszám: | 230 |
| Invoice date/Számla kelte: | 13.11.25 |
| Date of Perf./Teljesités dátuma: | 13.11.25 |
| Due date/Fizetési határidô: | 13.11.25 |
| Invoice No/Számlaszám: | 10401227126 |
| Reference/Ügyintézô: | Peter Vieland |

Name and address of partner/Vevô neve, címe:

Nemetschek Group SE

Konrad-Zuse-Platz 1

München, Bayern 81829 Germany

Tax code/Adószám:


| Date of | Service/Szolgáltatás | - |
| --- | --- | --- |
| services/Szolg. | Product/Termék | - |
| dátum | (Stat. No/SZJ Vat/Áfa %) | - |

Net/Nettó:

HUF

M. / Q.

M.e. / U. of q.

Unit Price Egységár

Net Flat Összesen

VAT Amount ÁFA összeg

Gross Value Bruttó Érték


| - | Nettó / Net | ÁFA/VAT | Bruttó / Gross |
| --- | --- | --- | --- |
| VAT 27%/ÁFA 27% | 0 | 0 | 0 |
| VAT 18%/ÁFA 18% | 0 | 0 | 0 |
| VAT 5%/ÁFA 5% | 232,050 | 11,602 | 243,652 |
| VAT 0%/ÁFAmentes | 0 | 0 | 0 |
| No VAT/ÁFA kör.kív. | 0 | 0 | 0 |
| Total/Összesen | 232,050 | 11,602 | 243,652 |
| City Taxl/IFA | 8,398 | 0 | 8,398 |
| Total/Mindösszesen | 240,448 | 11,602 | 252,050 |

Amount Due/Fizetendô (HUF)

0

TIP (HUF)

Comments/Megjegyzések:

Guest Signature/Vendég Aláírás

This invoice was created by Opera V5, Micros Fidelio Inc./A számla az Opera V5, Micros Fidelio Inc. rendszerével készült. Only amounts printed by the cashier are valid!/Csak a számlázógéppel nyomtatott összegek érvényesek! Valid without signature and stamp./Aláírás és pecsét nélkül érvényes. ÁKK – out of scope of VAT/ÁFA tv. hatályán kívül. ÁFA m. - exempt from VAT payment in line with Act on VAT/ÁFA tv. szerint áfa mentes. FAD - reverse charge procedure/Fordított adózás.

Merchant ID

Transaction ID

10453625

Approval Code/Eng.kód

220314

Approval Amount/Eng.összeg

252,050

Credit Card #/Kártyaszám:

Credit Card Expiry/Lejárat

XXXXXXXXXXXX9534 XX/XX

Capture Method:

Amount/Összeg

Chip 252,050

Please charge the above amount to my credit card/Kérem a fenti összeggel a hitelkártyámat megterhelni.

Danubius Hotels Zrt.

Danubius Hotel Helia

H-1133 Budapest, Kárpát u. 62-64.

Tax number/Adószám: 10594702-2-44

EU tax number/Közösségi adószám: HU10594702

Banknév és számlaszámok/Bank's name and accounts:

K&H Bank Zrt., H-1095 Budapest, Lechner Ödön fasor 9.

K&H EUR IBAN: HU29 1040 21424948 49565757 1817

K&H HUF: 10402142-49484956-57571721

SWIFT: OKHBHUHB


Wrapped up:

In [8]:
import os
from finance_analysis.config import global_config as glob   
from finance_analysis.resources.document_processor import DocumentProcessor_GCP

# invoice_name = "Free2move_DE_251113_4248127_3285000079177057.pdf"
# invoice_name = "Departure_taxi.pdf"
invoice_name = "Hotel_Invoice.pdf"

docs = DocumentProcessor_GCP(
    file_path=os.path.join(glob.DATA_PKG_DIR, invoice_name),
    project_id="neme-ai-rnd-dev-prj-01",
    location="eu",
    processor_id="c48f6c912b9ff9d5"
)
result = docs.process()

docs.to_markdown()

2025-11-16 01:13:42,372 - finance_analysis.resources.document_processor - INFO - ✅ Document processed successfully: Hotel_Invoice.pdf


"HOTELS\n\nDANUBIUS\n\nSzámla/Invoice\n\nOldal / Page\n\n1/2\n\n\n| VOSSELER, Name/Név: | ALEXANDER |\n| --- | --- |\n| Ref./Hiv.: | - |\n| Arrival/Érkezés: | 09.11.25 |\n| Departure/Utazás: | 13.11.25 |\n| Room No./Szobaszám: | 230 |\n| Invoice date/Számla kelte: | 13.11.25 |\n| Date of Perf./Teljesités dátuma: | 13.11.25 |\n| Due date/Fizetési határidô: | 13.11.25 |\n| Invoice No/Számlaszám: | 10401227126 |\n| Reference/Ügyintézô: | Peter Vieland |\n\nName and address of partner/Vevô neve, címe:\n\nNemetschek Group SE\n\nKonrad-Zuse-Platz 1\n\nMünchen, Bayern 81829 Germany\n\nTax code/Adószám:\n\n\n| Date of | Service/Szolgáltatás | Net/Nettó: | - | - | - | HUF |\n| --- | --- | --- | --- | --- | --- | --- |\n| services/Szolg. dátum | M. / Q. M.e. / U. of q. Product/Termék (Stat. No/SZJ Vat/Áfa %) | Unit Price Egységár | - | Net Flat Összesen | VAT Amount ÁFA összeg | Gross Value Bruttó Érték |\n| 09.11.25 | Visa Debit 1 db | - | - | - | - | -252,050 |\n| 09.11.25 | -Room/Szoba 55.10.

In [9]:
docs.display_markdown()

HOTELS

DANUBIUS

Számla/Invoice

Oldal / Page

1/2


| VOSSELER, Name/Név: | ALEXANDER |
| --- | --- |
| Ref./Hiv.: | - |
| Arrival/Érkezés: | 09.11.25 |
| Departure/Utazás: | 13.11.25 |
| Room No./Szobaszám: | 230 |
| Invoice date/Számla kelte: | 13.11.25 |
| Date of Perf./Teljesités dátuma: | 13.11.25 |
| Due date/Fizetési határidô: | 13.11.25 |
| Invoice No/Számlaszám: | 10401227126 |
| Reference/Ügyintézô: | Peter Vieland |

Name and address of partner/Vevô neve, címe:

Nemetschek Group SE

Konrad-Zuse-Platz 1

München, Bayern 81829 Germany

Tax code/Adószám:


| Date of | Service/Szolgáltatás | Net/Nettó: | - | - | - | HUF |
| --- | --- | --- | --- | --- | --- | --- |
| services/Szolg. dátum | M. / Q. M.e. / U. of q. Product/Termék (Stat. No/SZJ Vat/Áfa %) | Unit Price Egységár | - | Net Flat Összesen | VAT Amount ÁFA összeg | Gross Value Bruttó Érték |
| 09.11.25 | Visa Debit 1 db | - | - | - | - | -252,050 |
| 09.11.25 | -Room/Szoba 55.10.15% 1 db | 52,844 | - | 52,844 | 2,642 | 55,486 |
| 09.11.25 | -City Tax / IFA 1 db | 2,114 | - | 2,114 | - | 2,114 |
| 09.11.25 | -Package Breakfast/Reggeli 56.10.11 5% 1 db | 5,409 | - | 5,409 | 270 | 5,679 |
| 09.11.25 | -Service Charge/Felszolgálási díj 56.10.11 5% 1 db | 114 | - | 114 | 6 | 120 |
| 10.11.25 | -Room/Szoba 55.10.15% 1 db | 56,746 | - | 56,746 | 2,837 | 59,583 |
| 10.11.25 | -City Tax / IFA 1 db | 2,270 | - | 2,270 | - | 2,270 |
| 10.11.25 | -Package Breakfast/Reggeli 56.10.11 5% 1 db | 5,409 | - | 5,409 | 270 | 5,679 |
| 10.11.25 | -Service Charge/Felszolgálási díj 56.10.11 5% 1 db | 114 | - | 114 | 6 | 120 |
| 11.11.25 | -Room/Szoba 55.10.15% 1 db | 51,071 | - | 51,071 | 2,554 | 53,625 |
| 11.11.25 | -City Tax / IFA 1 db | 2,042 | - | 2,042 | - | 2,042 |
| 11.11.25 | -Package Breakfast/Reggeli 56.10.11 5% 1 db | 5,409 | - | 5,409 | 270 | 5,679 |
| 11.11.25 | -Service Charge/Felszolgálási díj 56.10.11 5% 1 db | 114 | - | 114 | 6 | 120 |
| 12.11.25 | -Room/Szoba 55.10.15% 1 db | 49,297 | - | 49,297 | 2,465 | 51,762 |
| 12.11.25 | -City Tax / IFA 1 db | 1,972 | - | 1,972 | - | 1,972 |
| 12.11.25 | -Package Breakfast/Reggeli 56.10.11 5% 1 db | 5,409 | - | 5,409 | 270 | 5,679 |
| 12.11.25 | -Service Charge/Felszolgálási díj 56.10.11 5% 1 db | 114 | - | 114 | 6 | 120 |

Danubius Hotels Zrt.

Danubius Hotel Helia

H-1133 Budapest, Kárpát u. 62-64.

Tax number/Adószám: 10594702-2-44

EU tax number/Közösségi adószám: HU10594702

Banknév és számlaszámok/Bank's name and accounts:

K&H Bank Zrt., H-1095 Budapest, Lechner Ödön fasor 9.

K&H EUR IBAN: HU29 1040 21424948 49565757 1817

K&H HUF: 10402142-49484956-57571721

SWIFT: OKHBHUHB

HOTELS

H DANUBIUS

Számla/Invoice

Oldal / Page

2/2


| VOSSELER, Name/Név: | ALEXANDER |
| --- | --- |
| Ref./Hiv.: | - |
| Arrival/Érkezés: | 09.11.25 |
| Departure/Utazás: | 13.11.25 |
| Room No./Szobaszám: | 230 |
| Invoice date/Számla kelte: | 13.11.25 |
| Date of Perf./Teljesités dátuma: | 13.11.25 |
| Due date/Fizetési határidô: | 13.11.25 |
| Invoice No/Számlaszám: | 10401227126 |
| Reference/Ügyintézô: | Peter Vieland |

Name and address of partner/Vevô neve, címe:

Nemetschek Group SE

Konrad-Zuse-Platz 1

München, Bayern 81829 Germany

Tax code/Adószám:


| Date of | Service/Szolgáltatás | - |
| --- | --- | --- |
| services/Szolg. | Product/Termék | - |
| dátum | (Stat. No/SZJ Vat/Áfa %) | - |

Net/Nettó:

HUF

M. / Q.

M.e. / U. of q.

Unit Price Egységár

Net Flat Összesen

VAT Amount ÁFA összeg

Gross Value Bruttó Érték


| - | Nettó / Net | ÁFA/VAT | Bruttó / Gross |
| --- | --- | --- | --- |
| VAT 27%/ÁFA 27% | 0 | 0 | 0 |
| VAT 18%/ÁFA 18% | 0 | 0 | 0 |
| VAT 5%/ÁFA 5% | 232,050 | 11,602 | 243,652 |
| VAT 0%/ÁFAmentes | 0 | 0 | 0 |
| No VAT/ÁFA kör.kív. | 0 | 0 | 0 |
| Total/Összesen | 232,050 | 11,602 | 243,652 |
| City Taxl/IFA | 8,398 | 0 | 8,398 |
| Total/Mindösszesen | 240,448 | 11,602 | 252,050 |

Amount Due/Fizetendô (HUF)

0

TIP (HUF)

Comments/Megjegyzések:

Guest Signature/Vendég Aláírás

This invoice was created by Opera V5, Micros Fidelio Inc./A számla az Opera V5, Micros Fidelio Inc. rendszerével készült. Only amounts printed by the cashier are valid!/Csak a számlázógéppel nyomtatott összegek érvényesek! Valid without signature and stamp./Aláírás és pecsét nélkül érvényes. ÁKK – out of scope of VAT/ÁFA tv. hatályán kívül. ÁFA m. - exempt from VAT payment in line with Act on VAT/ÁFA tv. szerint áfa mentes. FAD - reverse charge procedure/Fordított adózás.

Merchant ID

Transaction ID

10453625

Approval Code/Eng.kód

220314

Approval Amount/Eng.összeg

252,050

Credit Card #/Kártyaszám:

Credit Card Expiry/Lejárat

XXXXXXXXXXXX9534 XX/XX

Capture Method:

Amount/Összeg

Chip 252,050

Please charge the above amount to my credit card/Kérem a fenti összeggel a hitelkártyámat megterhelni.

Danubius Hotels Zrt.

Danubius Hotel Helia

H-1133 Budapest, Kárpát u. 62-64.

Tax number/Adószám: 10594702-2-44

EU tax number/Közösségi adószám: HU10594702

Banknév és számlaszámok/Bank's name and accounts:

K&H Bank Zrt., H-1095 Budapest, Lechner Ödön fasor 9.

K&H EUR IBAN: HU29 1040 21424948 49565757 1817

K&H HUF: 10402142-49484956-57571721

SWIFT: OKHBHUHB
